# ASL Translator - Improved Checkpoint 1 Notebook

This notebook is a cleaned and improved version of the original Checkpoint 1 notebook. It keeps the original idea, but fixes the questionable parts and connects the work to the Checkpoint 2 project pipeline.

## What Was Improved From Checkpoint 1

| Original issue | Improvement |
| --- | --- |
| Mixed TensorFlow/Keras and PyTorch code | The cleaned pipeline uses PyTorch consistently |
| Keras-style `model.evaluate()` and `model.predict()` used on a PyTorch model | Evaluation is moved to PyTorch scripts |
| Hardcoded local Windows dataset path | Dataset path is configurable |
| No saved model artifact | Training saves `models/asl_model.pt` |
| Only one model | Checkpoint 2 compares Simple CNN, ResNet18, and MobileNetV3 Small |
| No deployment connection | Saved model is loaded by FastAPI backend and React UI |
| Limited UI | UI supports webcam and image upload prediction |

## Project Structure

The notebook is now part of a full project:

```text
asl-translator/
  backend/      FastAPI inference API
  frontend/     React UI
  scripts/      training and comparison scripts
  models/       trained checkpoints and metrics
  notebooks/    original and improved notebooks
  docs/         report and presentation materials
```

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path('/Users/j-19group/asl-translator')
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)
PROJECT_ROOT

## Dataset Configuration

Set this path to your dataset. It can point either to the parent folder or directly to `asl_alphabet_train`.

Expected layout:

```text
asl_alphabet_train/
  A/
  B/
  ...
  Z/
  del/
  nothing/
  space/
```

In [ ]:
DATASET_PATH = '/path/to/ASL_Alphabet_Dataset'
EPOCHS = 5
BATCH_SIZE = 64
IMG_SIZE = 96
DATASET_PATH

## Baseline Model: Improved Simple CNN

This is the cleaned baseline model based on the original Checkpoint 1 idea. It uses the shared training script so evaluation and saving are consistent.

In [ ]:
!python {PROJECT_ROOT / 'scripts' / 'train.py'} --dataset "{DATASET_PATH}" --architecture simple_cnn --epochs {EPOCHS} --batch-size {BATCH_SIZE} --img-size {IMG_SIZE} --output {MODELS_DIR / 'simple_cnn.pt'}

## Model Comparison For Checkpoint 2

Checkpoint 2 requires comparison with at least two other models. We compare:

1. Simple CNN - baseline from Checkpoint 1
2. ResNet18 - deeper residual CNN
3. MobileNetV3 Small - efficient deployment-friendly CNN

In [ ]:
!python {PROJECT_ROOT / 'scripts' / 'compare_models.py'} --dataset "{DATASET_PATH}" --epochs {EPOCHS} --batch-size {BATCH_SIZE} --img-size {IMG_SIZE} --pretrained

If internet access is unavailable or pretrained weights cannot be downloaded, run without `--pretrained`:

```python
!python scripts/compare_models.py --dataset "..." --epochs 5
```

## Load And Display Comparison Results

In [ ]:
comparison_path = MODELS_DIR / 'model_comparison.json'
comparison = json.loads(comparison_path.read_text())
comparison_df = pd.DataFrame(comparison)
comparison_df

In [ ]:
comparison_df.plot.bar(
    x='architecture',
    y=['best_val_accuracy', 'test_accuracy'],
    figsize=(9, 4),
    ylim=(0, 1),
    title='Validation and Test Accuracy by Model'
);

In [ ]:
comparison_df.plot.bar(
    x='architecture',
    y='parameters',
    figsize=(9, 4),
    title='Model Complexity: Trainable Parameters'
);

## Error Analysis

The training script saves per-class accuracy and the most common mistakes for each model.

In [ ]:
best_metrics_path = Path(comparison_df.iloc[0]['metrics'])
best_metrics = json.loads(best_metrics_path.read_text())
print('Best model:', best_metrics['architecture'])
print('Best validation accuracy:', best_metrics['best_val_accuracy'])
print('Test accuracy:', best_metrics['test']['accuracy'])

In [ ]:
pd.DataFrame(best_metrics['test']['top_mistakes'])

In [ ]:
pd.DataFrame(best_metrics['test']['per_class']).sort_values('accuracy').head(10)

## Result Explanation Template

Fill this after training:

- The best model was: `...`
- It achieved test accuracy: `...`
- Compared with the baseline Simple CNN, it improved by: `...`
- The hardest classes were: `...`
- The most common mistakes were: `...`
- These mistakes likely happen because: similar hand shapes, lighting, hand position, background, or static representation of dynamic signs.
- For deployment, we choose: `...`, because it balances accuracy, speed, and model size.

## Connect Best Model To UI

After choosing the best checkpoint, copy it to the backend default model path:

```bash
cp models/resnet18.pt models/asl_model.pt
```

or use the best model filename from `model_comparison.json`.

## Final Checkpoint 2 Claim

Checkpoint 1 implemented a basic ASL alphabet classifier. Checkpoint 2 improves it by fixing the training/evaluation pipeline, adding model export, comparing three model architectures, adding model complexity analysis, connecting the model to a UI, and preparing documentation for report and presentation.